<a href="https://colab.research.google.com/github/guupiii/ESAA/blob/main/0306_%EC%84%B8%EC%85%98_%EB%AA%A8%EB%8D%B8%ED%9B%88%EB%A0%A8_%EC%97%B0%EC%8A%B5%EB%AC%B8%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **모델 훈련 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch04 연습문제 1, 5, 9, 10
- 개념 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

### **1. 수백만 개의 특성을 가진 훈련 세트에서는 어떤 선형 회귀 알고리즘을 사용할 수 있을까요?**
___


계산량과 메모리 요구가 매우 커지기 때문에 계산 효율이 높은 경사 하강법 기반 알고리즘을 사용할 수 있다. 확률적 경사 하강법(SGD)이나 미니배치 경사 하강법이 적합하다.

### **2. 배치 경사 하강법을 사용하고 에포크마다 검증 오차를 그래프로 나타내봤습니다. 검증 오차가 일정하게 상승되고 있다면 어떤 일이 일어나고 있는 걸까요? 이 문제를 어떻게 해결할 수 있나요?**
___

모델이 훈련 데이터에 과도하게 맞춰지는 과대적합이 발생하고 있을 가능성이 있다.  이를 해결하기 위해서는 Early Stopping를 사용하여 검증 오차가 증가하기 시작하는 시점에서 학습을 멈추거나, 규제를 적용하는 등의 방법을 사용해 볼 수 있다.

### **3. 릿지 회귀를 사용했을 때 훈련 오차가 검증 오차가 거의 비슷하고 둘 다 높았습니다. 이 모델에는 높은 편향이 문제인가요, 아니면 높은 분산이 문제인가요? 규제 하이퍼파라미터 $\alpha$를 증가시켜야 할까요 아니면 줄여야 할까요?**
___

과소적합 상태이고 높은 편향이 문제이다. 릿지 회귀의 규제 하이퍼파라미터
알파가 클수록 모델이 더 단순해지므로 편향이 더 커질 수 있다. 따라서 이러한 경우에는 알파값을 줄여야 한다.

### **4. 다음과 같이 사용해야 하는 이유는?**
___
- 평범한 선형 회귀(즉, 아무런 규제가 없는 모델) 대신 릿지 회귀
- 릿지 회귀 대신 라쏘 회귀
- 라쏘 회귀 대신 엘라스틱넷

- 규제를 통해 모델의 복잡도를 줄이고 과대적합을 방지하기 위해서이다.
- 라쏘가 일부 특성의 계수를 정확히 0으로 만들어 자동으로 특성 선택(feature selection)을 수행할 수 있기 때문이다.
- 라쏘가 서로 강하게 상관된 특성들 중 하나만 선택하는 경향이 있기 때문에, 릿지와 라쏘의 장점을 함께 사용하는 엘라스틱넷을 통해 규제와 특성 선택을 동시에 안정적으로 수행할 수 있다.

### **추가) 조기 종료를 사용한 배치 경사 하강법으로 iris 데이터를 활용해 소프트맥스 회귀를 구현해보세요(사이킷런은 사용하지 마세요)**


---



In [1]:
import numpy as np
from sklearn.datasets import load_iris

# 데이터
iris = load_iris()
X = iris.data
y = iris.target
Y = np.eye(3)[y]

X_train, X_val = X[:120], X[120:]
Y_train, Y_val = Y[:120], Y[120:]

# 초기화
W = np.random.randn(4,3)
lr = 0.01
best_loss = np.inf

def softmax(z):
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

for epoch in range(1000):

    # 예측
    P = softmax(X_train @ W)

    # gradient
    grad = X_train.T @ (P - Y_train) / len(X_train)

    # 업데이트
    W -= lr * grad

    # 검증 오차
    val_pred = softmax(X_val @ W)
    val_loss = -np.mean(np.sum(Y_val*np.log(val_pred), axis=1))

    # Early stopping
    if val_loss > best_loss:
        break
    best_loss = val_loss